**Rules for the code:**

- Include all the code you used for your report in this file. The code for any section in the report should go under the same section in this file.
- Any missing code will result in -20% from its corresponding section in the report.
- Any irrelevant code will result in -20% from its corresponding section in the report.
- Make sure that you run your code before rendering so that all the necessary visual/numeric outputs are visible.
- Any code that is not properly run or throws errors will be considered missing/irrelevant.

## 4) Data

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.formula.api as smf
from statsmodels.tools.tools import add_constant

from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Lasso, Ridge, LassoCV, RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

data = pd.read_csv('ca_real_estate.csv')

num_rows, num_cols = data.shape
print(f"\nNumber of observations (rows): {num_rows}")
print(f"Number of variables (columns): {num_cols}")

continuous_vars = data.select_dtypes(include=[np.number]).columns.tolist()
categorical_vars = data.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"\nNumber of continuous variables: {len(continuous_vars)}")
print(f"Number of categorical variables: {len(categorical_vars)}")

missing_values = data.isnull().sum()
print("\nMissing Values Summary:")
print(missing_values)
print('\nBecause all values are found, cleaning/imputing is not necessary')


Number of observations (rows): 5000
Number of variables (columns): 10

Number of continuous variables: 7
Number of categorical variables: 3

Missing Values Summary:
Price         0
Bedrooms      0
Bathrooms     0
SqFt          0
City          0
Province      0
Year_Built    0
Type          0
Garage        0
Lot_Area      0
dtype: int64

Because all values are found, cleaning/imputing is not necessary


## 5) Prediction

In [5]:
y = data['Price']
X = data[['Bedrooms', 'Bathrooms', 'SqFt', 'City', 'Province', 'Year_Built', 'Type', 'Garage', 'Lot_Area']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

In [6]:
def compute_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def get_train_test_rmse(formula, train_df, test_df, y_col='Price'):
    model = smf.ols(formula=formula, data=train_df).fit()
    y_train_pred = model.predict(train_df)
    y_test_pred  = model.predict(test_df)
    rmse_train = compute_rmse(train_df[y_col], y_train_pred)
    rmse_test  = compute_rmse(test_df[y_col],  y_test_pred)
    return rmse_train, rmse_test

train_df = X_train.copy()
train_df['Price'] = y_train

test_df = X_test.copy()
test_df['Price'] = y_test

predictors_in_order = [
    ['SqFt'],  
    ['SqFt', 'Bedrooms'],
    ['SqFt', 'Bedrooms', 'Bathrooms'],
    ['SqFt', 'Bedrooms', 'Bathrooms', 'Year_Built'],
    ['SqFt', 'Bedrooms', 'Bathrooms', 'Year_Built', 'Lot_Area'],
    ['SqFt', 'Bedrooms', 'Bathrooms', 'Year_Built', 'Lot_Area', 'C(City)'],
    ['SqFt', 'Bedrooms', 'Bathrooms', 'Year_Built', 'Lot_Area', 'C(City)', 'C(Province)'],
    ['SqFt', 'Bedrooms', 'Bathrooms', 'Year_Built', 'Lot_Area', 'C(City)', 'C(Province)', 'C(Type)'],
    ['SqFt', 'Bedrooms', 'Bathrooms', 'Year_Built', 'Lot_Area', 'C(City)', 'C(Province)',
     'C(Type)', 'Garage']
]

results = []
for step_preds in predictors_in_order:
    formula_str = "Price ~ " + " + ".join(step_preds)
    rmse_train, rmse_test = get_train_test_rmse(formula_str, train_df, test_df, 'Price')
    results.append({
        'Predictors': step_preds,
        'Train_RMSE': rmse_train,
        'Test_RMSE': rmse_test
    })

results_df = pd.DataFrame(results)
print("\n=== Stepwise addition of predictors (unregularized) ===")
results_df.head(10)


=== Stepwise addition of predictors (unregularized) ===


,Predictors,Train_RMSE,Test_RMSE
0,[SqFt],259519.721356,256901.538533
1,"[SqFt, Bedrooms]",259278.861555,257266.070031
2,"[SqFt, Bedrooms, Bathrooms]",259278.802886,257267.964925
3,"[SqFt, Bedrooms, Bathrooms, Year_Built]",259269.709495,257256.716240
4,"[SqFt, Bedrooms, Bathrooms, Year_Built, Lot_Area]",259245.450866,257426.132401
5,"[SqFt, Bedrooms, Bathrooms, Year_Built, Lot_Ar...",259198.100609,257543.209325
6,"[SqFt, Bedrooms, Bathrooms, Year_Built, Lot_Ar...",259056.053642,257478.854179
7,"[SqFt, Bedrooms, Bathrooms, Year_Built, Lot_Ar...",258848.195340,257929.608778
8,"[SqFt, Bedrooms, Bathrooms, Year_Built, Lot_Ar...",258847.602698,257934.279403


In [7]:
num_cols = ['Bedrooms', 'Bathrooms', 'SqFt', 'Year_Built', 'Garage', 'Lot_Area']
cat_cols = ['City', 'Province', 'Type'] 

# OHE
X_train_num = X_train[num_cols]
X_test_num  = X_test[num_cols]

X_train_cat = pd.get_dummies(X_train[cat_cols], drop_first=True)
X_test_cat  = pd.get_dummies(X_test[cat_cols], drop_first=True)

X_test_cat = X_test_cat.reindex(columns=X_train_cat.columns, fill_value=0)

poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_num) 
X_test_poly  = poly.transform(X_test_num)

poly_feature_names = poly.get_feature_names_out(num_cols)

X_train_expanded = np.concatenate([X_train_poly, X_train_cat.values], axis=1)
X_test_expanded  = np.concatenate([X_test_poly,  X_test_cat.values ], axis=1)

scaler = StandardScaler()
scaler.fit(X_train_expanded)
X_train_scaled = scaler.transform(X_train_expanded)
X_test_scaled  = scaler.transform(X_test_expanded)

In [8]:
# Perform ridgeCV to see if better
ridge_cv = RidgeCV(alphas=10**np.linspace(5, -2, 100)*0.5, cv=5)
ridge_cv.fit(X_train_scaled, y_train)
print("Best Ridge Alpha:", ridge_cv.alpha_)
ridge_test_rmse = np.sqrt(mean_squared_error(y_test, ridge_cv.predict(X_test_scaled)))
print("Ridge Test RMSE:", ridge_test_rmse)

# Perform Lasscv to see if better
alphas_to_try = 10**np.linspace(5, -2, 200) * 0.5 
model_cv = LassoCV(alphas=alphas_to_try, cv=5, max_iter=200_000)
model_cv.fit(X_train_scaled, y_train)
print("\nBest Lasso Alpha:", model_cv.alpha_)
y_test_pred_cv = model_cv.predict(X_test_scaled)
test_rmse_cv = np.sqrt(mean_squared_error(y_test, y_test_pred_cv))
print("Test RMSE (with best alpha using Lasso) =", test_rmse_cv)

Best Ridge Alpha: 15996.335688986923
Ridge Test RMSE: 257332.55500989687

Best Lasso Alpha: 7761.12678713524
Test RMSE (with best alpha using Lasso) = 257214.5198652233


In [9]:
corr_matrix = data[continuous_vars].corr()
print("\nCorrelation Matrix (Continuous Variables):")
corr_matrix


Correlation Matrix (Continuous Variables):


,Price,Bedrooms,Bathrooms,SqFt,Year_Built,Garage,Lot_Area
Price,1.000000,-0.032169,-0.002194,-0.030875,0.009489,0.000591,-0.002460
Bedrooms,-0.032169,1.000000,0.018430,-0.004158,-0.001657,0.006680,-0.010404
Bathrooms,-0.002194,0.018430,1.000000,0.006462,0.009065,0.002102,-0.008651
SqFt,-0.030875,-0.004158,0.006462,1.000000,-0.024400,-0.006923,-0.001564
Year_Built,0.009489,-0.001657,0.009065,-0.024400,1.000000,0.012056,-0.004763
Garage,0.000591,0.006680,0.002102,-0.006923,0.012056,1.000000,-0.014666
Lot_Area,-0.002460,-0.010404,-0.008651,-0.001564,-0.004763,-0.014666,1.000000


In [10]:
all_cat_dummies = list(X_train_cat.columns)  # these come after the poly numeric features
all_feature_names = list(poly_feature_names) + all_cat_dummies

coef_array = model_cv.coef_
coef_df = pd.DataFrame({
    'feature': all_feature_names,
    'coefficient': coef_array
})
coef_df['abs_coef'] = coef_df['coefficient'].abs()
coef_df.sort_values('abs_coef', ascending=False, inplace=True)

print("\n--- Top 5 Largest Coefficients (by absolute value) ---")
print(coef_df.head(5)[['feature','coefficient']])


--- Top 5 Largest Coefficients (by absolute value) ---
              feature  coefficient
8       Bedrooms SqFt -5585.853717
35         Type_House -1852.924357
27      City_Montreal     0.000000
21       Year_Built^2     0.000000
22  Year_Built Garage     0.000000


In [11]:
print("\n--- Bottom 5 (smallest absolute value) ---")
print(coef_df.tail(5)[['feature','coefficient']])


--- Bottom 5 (smallest absolute value) ---
                 feature  coefficient
13        Bathrooms SqFt         -0.0
14  Bathrooms Year_Built          0.0
15      Bathrooms Garage          0.0
16    Bathrooms Lot_Area         -0.0
18       SqFt Year_Built         -0.0


In [12]:
zero_mask = (coef_df['coefficient'] == 0.0)
zeros_df = coef_df[zero_mask]
if len(zeros_df) > 0:
    print(f"\nNumber of zero-coeff terms: {len(zeros_df)}")
    print(zeros_df[['feature','coefficient']])


Number of zero-coeff terms: 34
                 feature  coefficient
27         City_Montreal          0.0
21          Year_Built^2          0.0
22     Year_Built Garage          0.0
23   Year_Built Lot_Area         -0.0
24              Garage^2          0.0
25       Garage Lot_Area          0.0
26            Lot_Area^2         -0.0
28           City_Ottawa         -0.0
19           SqFt Garage          0.0
29          City_Toronto          0.0
30        City_Vancouver          0.0
31           Province_BC          0.0
32           Province_ON          0.0
33           Province_QC         -0.0
34            Type_Condo          0.0
20         SqFt Lot_Area         -0.0
0               Bedrooms         -0.0
1              Bathrooms         -0.0
9    Bedrooms Year_Built         -0.0
2                   SqFt         -0.0
3             Year_Built          0.0
4                 Garage          0.0
5               Lot_Area         -0.0
6             Bedrooms^2         -0.0
7     Bedrooms Bat

## 6) Inference

In [14]:
final_formula = """
Price ~ SqFt
       + Bedrooms
       + Bathrooms
       + Year_Built
       + C(Type)
       + I(Bedrooms**2)
       + Bedrooms:Bathrooms
       + Bedrooms:SqFt
       + C(City)
       + C(Province)
"""

final_model = smf.ols(final_formula, data=train_df).fit()
print("\n--- FINAL MODEL SUMMARY (statsmodels) ---")
print(final_model.summary())

final_preds = final_model.predict(test_df)
final_rmse = np.sqrt(mean_squared_error(test_df['Price'], final_preds))
print("\nFinal model test RMSE =", final_rmse)


--- FINAL MODEL SUMMARY (statsmodels) ---
                            OLS Regression Results                            
Dep. Variable:                  Price   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     1.691
Date:                Fri, 21 Mar 2025   Prob (F-statistic):             0.0414
Time:                        22:26:36   Log-Likelihood:                -55529.
No. Observations:                4000   AIC:                         1.111e+05
Df Residuals:                    3983   BIC:                         1.112e+05
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------